## Build a RAG agent with LangChain
https://docs.langchain.com/oss/python/langchain/rag

### LangSmith
Many of the applications you build with LangChain will contain multiple steps with multiple invocations of LLM calls. As these applications get more complex, it becomes crucial to be able to inspect what exactly is going on inside your chain or agent. The best way to do this is with LangSmith.

After you sign up at the link above, make sure to set your environment variables to start logging traces:

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

### Components

Select a chat model - Groq

In [2]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.2,
)

Select an embeddings model - HuggingFace

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Select a vector store - Chroma

In [4]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db", # Where to save data locally, remove if not necessary
)

## 1. Indexing
Indexing commonly works as follows:

Load: First we need to load our data. This is done with Document Loaders.

Split: Text splitters break large Documents into smaller chunks. This is useful both for indexing data and passing it into a model, as large chunks are harder to search over and won’t fit in a model’s finite context window.

Store: We need somewhere to store and index our splits, so that they can be searched over later. This is often done using a VectorStore and Embeddings model.

#### Loading documents

In [5]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(
    class_=("post-title", "post-header", "post-content")
)
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)

docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Total characters: 43047


In [6]:
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


#### Splitting documents

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,         # chunk size (characters)
    chunk_overlap=200,       # chunk overlap (characters)
    add_start_index=True,    # track index in original documents
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 63 sub-documents.


#### Storing documents

In [8]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['b65116e5-dc6e-4964-b771-ab9eaff7cc5e', 'e2e6548e-eb69-4278-87ce-ae8e5bc15659', 'bd126617-ac7d-4db3-ae01-5a25b26d82d2']


## 2. Retrieval and generation

#### RAG agents

In [9]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_content(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [10]:
from langchain.agents import create_agent

tools = [retrieve_content]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a blog post."
    "Use the tool to help answer these user queries."
)

agent = create_agent(model, tools, system_prompt=prompt)

In [11]:
query = (
    "What is the standard method for Task Decomposition?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

for event in agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================
Tool Calls:
  retrieve_content (qmq27xvr1)
 Call ID: qmq27xvr1
  Args:
    query: What is the standard method for Task Decomposition?
================================= Tool Message =================================
Name: retrieve_content

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 2578}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-h

#### RAG Chains

In [12]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    """Inject context into state messages."""
    last_query = request.state["messages"][-1].text
    retrieved_docs = vector_store.similarity_search(last_query)

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    system_message = (
        "You are a helpful assistant. Use the following context in your response"
        f"\n\n{docs_content}"
    )

    return system_message

agent = create_agent(model, tools=[], middleware=[prompt_with_context])

In [13]:
query = "What is task decomposition?"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is task decomposition?
================================== Ai Message ==================================

<think>
Okay, the user is asking, "What is task decomposition?" Let me start by recalling the information provided in the context. The context mentions that task decomposition can be done in three ways: using LLM with simple prompts, task-specific instructions, or human inputs. There's also the LLM+P approach which uses PDDL and a classical planner.

First, I need to define task decomposition in simple terms. It's breaking down a complex task into smaller, manageable subtasks. The user might be someone looking to understand how to approach a big project or problem, maybe a student or a project manager. They might want to know the methods available and when to use them.

The context repeats the same information multiple times, so I should make sure not to repeat the same points. The key points are 